In [2]:
%cd OneFormer/
import numpy as np
import cv2
import torch
import cv2
import imutils

import sys, os, distutils.core
# !git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
# !python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])} --quiet
sys.path.insert(0, os.path.abspath('./detectron2'))

/home/sabbbir/Documents/of/OneFormer


In [3]:
######
#@title 3. Import Libraries and other Utilities
######
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
setup_logger(name="oneformer")

# Import libraries
import numpy as np
import cv2
import torch
import cv2
import imutils

# Import detectron2 utilities
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.data import MetadataCatalog
from demo.defaults import DefaultPredictor
from demo.visualizer import Visualizer, ColorMode


# import OneFormer Project
from oneformer import (
    add_oneformer_config,
    add_common_config,
    add_swin_config,
    add_dinat_config,
    add_convnext_config,
)

######
#@title 4. Define helper functions
######
cpu_device = torch.device("cpu")
SWIN_CFG_DICT = {"cityscapes": "configs/cityscapes/oneformer_swin_large_IN21k_384_bs16_90k.yaml",
            "coco": "configs/coco/oneformer_swin_large_IN21k_384_bs16_100ep.yaml",
            "ade20k": "configs/ade20k/oneformer_swin_large_IN21k_384_bs16_160k.yaml",}

DINAT_CFG_DICT = {"cityscapes": "configs/cityscapes/oneformer_dinat_large_bs16_90k.yaml",
            "coco": "configs/coco/oneformer_dinat_large_bs16_100ep.yaml",
            "ade20k": "configs/ade20k/oneformer_dinat_large_IN21k_384_bs16_160k.yaml",}

def setup_cfg(dataset, model_path, use_swin):
    # load config from file and command-line arguments
    cfg = get_cfg()
    add_deeplab_config(cfg)
    add_common_config(cfg)
    add_swin_config(cfg)
    add_dinat_config(cfg)
    add_convnext_config(cfg)
    add_oneformer_config(cfg)
    if use_swin:
      cfg_path = SWIN_CFG_DICT[dataset]
    else:
      cfg_path = DINAT_CFG_DICT[dataset]
    cfg.merge_from_file(cfg_path)
    cfg.MODEL.DEVICE = 'cuda:0'
    cfg.MODEL.WEIGHTS = model_path
    cfg.freeze()
    return cfg

def setup_modules(dataset, model_path, use_swin):
    cfg = setup_cfg(dataset, model_path, use_swin)
    predictor = DefaultPredictor(cfg)
    metadata = MetadataCatalog.get(
        cfg.DATASETS.TEST_PANOPTIC[0] if len(cfg.DATASETS.TEST_PANOPTIC) else "__unused"
    )
    if 'cityscapes_fine_sem_seg_val' in cfg.DATASETS.TEST_PANOPTIC[0]:
        from cityscapesscripts.helpers.labels import labels
        stuff_colors = [k.color for k in labels if k.trainId != 255]
        metadata = metadata.set(stuff_colors=stuff_colors)

    return predictor, metadata

def panoptic_run(img, predictor, metadata):
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, instance_mode=ColorMode.IMAGE)
    predictions = predictor(img, "panoptic")
    panoptic_seg, segments_info = predictions["panoptic_seg"]
    out = visualizer.draw_panoptic_seg_predictions(
    panoptic_seg.to(cpu_device), segments_info, alpha=0.5
)
    return out

def instance_run(img, predictor, metadata):
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, instance_mode=ColorMode.IMAGE)
    predictions = predictor(img, "instance")
    instances = predictions["instances"].to(cpu_device)
    out = visualizer.draw_instance_predictions(predictions=instances, alpha=0.5)
    return out

def semantic_run(img, predictor, metadata):
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, instance_mode=ColorMode.IMAGE)
    predictions = predictor(img, "semantic")
    out = visualizer.draw_sem_seg(
        predictions["sem_seg"].argmax(dim=0).to(cpu_device), alpha=0.5
    )
    return out

TASK_INFER = {"panoptic": panoptic_run,
              "instance": instance_run,
              "semantic": semantic_run}

Popen(['git', 'version'], cwd=/home/sabbbir/Documents/of/OneFormer, stdin=None, shell=False, universal_newlines=False)
Popen(['git', 'version'], cwd=/home/sabbbir/Documents/of/OneFormer, stdin=None, shell=False, universal_newlines=False)
Trying paths: ['/home/sabbbir/.docker/config.json', '/home/sabbbir/.dockercfg']
No config file found
Setting up integrations (with default = False)
Setting SDK name to 'sentry.python'
[Profiling] Setting up continuous profiler in thread mode


/home/sabbbir/miniconda3/envs/oneformer/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sabbbir/Documents/of/OneFormer/oneformer/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


In [4]:
import os
import subprocess

use_swin = True #@param {type: 'boolean'}

def city():
  """
  Initializes the OneFormer model with either DiNAT-L or Swin-L backbone
  based on the 'use_swin' global variable. Downloads the corresponding
  checkpoint if it doesn't exist.
  """
  global use_swin
  if not use_swin:
    checkpoint_path = "250_16_dinat_l_oneformer_cityscapes_90k.pth"
    checkpoint_url = "https://shi-labs.com/projects/oneformer/cityscapes/250_16_dinat_l_oneformer_cityscapes_90k.pth"
  else:
    checkpoint_path = "250_16_swin_l_oneformer_cityscapes_90k.pth"
    checkpoint_url = "https://shi-labs.com/projects/oneformer/cityscapes/250_16_swin_l_oneformer_cityscapes_90k.pth"

  if not os.path.exists(checkpoint_path):
    print(f"Downloading checkpoint: {checkpoint_path}...")
    subprocess.run(f'wget {checkpoint_url}', shell=True)
    print("Download complete.")
  else:
    print(f"Checkpoint already exists: {checkpoint_path}")

  predictor, metadata = setup_modules("cityscapes", checkpoint_path, use_swin)
  return predictor, metadata

# Example of how to call the function:
predictor, metadata = city()


def ade():
  """
  Initializes the OneFormer model for the ADE20K dataset.
  Downloads the checkpoint if it doesn't exist.
  Assumes success of download and setup.

  Returns:
      tuple: (predictor, metadata) - The initialized predictor and metadata.  May return invalid values if download or setup fails.
  """
  global use_swin
  if not use_swin:
    checkpoint_path = "250_16_dinat_l_oneformer_ade20k_160k.pth"
    checkpoint_url = "https://shi-labs.com/projects/oneformer/ade20k/250_16_dinat_l_oneformer_ade20k_160k.pth"
  else:
    checkpoint_path = "250_16_swin_l_oneformer_ade20k_160k.pth"
    checkpoint_url = "https://shi-labs.com/projects/oneformer/ade20k/250_16_swin_l_oneformer_ade20k_160k.pth"

  if not os.path.exists(checkpoint_path):
    print(f"Downloading checkpoint: {checkpoint_path}...")
    subprocess.run(['wget', checkpoint_url], check=True)
    print("Download complete.")

  predictor, metadata = setup_modules("ade20k", checkpoint_path, use_swin)
  return predictor, metadata


predictor, metadata = ade()



def coco():
  global use_swin
  if not use_swin:
    checkpoint_path = "150_16_dinat_l_oneformer_coco_100ep.pth"
    checkpoint_url = 'wget https://shi-labs.com/projects/oneformer/coco/150_16_dinat_l_oneformer_coco_100ep.pth'
  else:
    checkpoint_path = "150_16_swin_l_oneformer_coco_100ep.pth"
    checkpoint_url = "wget https://shi-labs.com/projects/oneformer/coco/150_16_swin_l_oneformer_coco_100ep.pth"

  if not os.path.exists(checkpoint_path):
    print(f"Downloading checkpoint: {checkpoint_path}...")
    subprocess.run(f'wget {checkpoint_url}', shell=True)
    print("Download complete.")
  else:
    print(f"Checkpoint already exists: {checkpoint_path}")

    predictor, metadata = setup_modules("coco", checkpoint_path, use_swin)
    return predictor, metadata



predictor, metadata = coco()

    # You would use predictor and metadata here.  No error checking.

# Now 'predictor' and 'metadata' are initialized and ready to use.
# Now 'predictor' and 'metadata' are initialized and ready to use.

Loading config configs/cityscapes/Base-Cityscapes-UnifiedSegmentation.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Checkpoint already exists: 250_16_swin_l_oneformer_cityscapes_90k.pth


/home/sabbbir/miniconda3/envs/oneformer/lib/python3.12/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[04/29 10:05:30 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from 250_16_swin_l_oneformer_cityscapes_90k.pth ...
[Checkpointer] Loading from 250_16_swin_l_oneformer_cityscapes_90k.pth ...


/home/sabbbir/miniconda3/envs/oneformer/lib/python3.12/site-packages/fvcore/common/checkpoint.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_

[04/29 10:05:33 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from 250_16_swin_l_oneformer_ade20k_160k.pth ...
[Checkpointer] Loading from 250_16_swin_l_oneformer_ade20k_160k.pth ...


The checkpoint state_dict contains keys that are not used by the model:
  text_encoder.positional_embedding
  text_encoder.transformer.resblocks.0.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.0.attn.out_proj.{bias, weight}
  text_encoder.transformer.resblocks.0.ln_1.{bias, weight}
  text_encoder.transformer.resblocks.0.mlp.c_fc.{bias, weight}
  text_encoder.transformer.resblocks.0.mlp.c_proj.{bias, weight}
  text_encoder.transformer.resblocks.0.ln_2.{bias, weight}
  text_encoder.transformer.resblocks.1.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.1.attn.out_proj.{bias, weight}
  text_encoder.transformer.resblocks.1.ln_1.{bias, weight}
  text_encoder.transformer.resblocks.1.mlp.c_fc.{bias, weight}
  text_encoder.transformer.resblocks.1.mlp.c_proj.{bias, weight}
  text_encoder.transformer.resblocks.1.ln_2.{bias, weight}
  text_encoder.transformer.resblocks.2.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.2.

Checkpoint already exists: 150_16_swin_l_oneformer_coco_100ep.pth
[04/29 10:05:35 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from 150_16_swin_l_oneformer_coco_100ep.pth ...
[Checkpointer] Loading from 150_16_swin_l_oneformer_coco_100ep.pth ...


The checkpoint state_dict contains keys that are not used by the model:
  text_encoder.positional_embedding
  text_encoder.transformer.resblocks.0.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.0.attn.out_proj.{bias, weight}
  text_encoder.transformer.resblocks.0.ln_1.{bias, weight}
  text_encoder.transformer.resblocks.0.mlp.c_fc.{bias, weight}
  text_encoder.transformer.resblocks.0.mlp.c_proj.{bias, weight}
  text_encoder.transformer.resblocks.0.ln_2.{bias, weight}
  text_encoder.transformer.resblocks.1.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.1.attn.out_proj.{bias, weight}
  text_encoder.transformer.resblocks.1.ln_1.{bias, weight}
  text_encoder.transformer.resblocks.1.mlp.c_fc.{bias, weight}
  text_encoder.transformer.resblocks.1.mlp.c_proj.{bias, weight}
  text_encoder.transformer.resblocks.1.ln_2.{bias, weight}
  text_encoder.transformer.resblocks.2.attn.{in_proj_bias, in_proj_weight}
  text_encoder.transformer.resblocks.2.

In [ ]:
predictor, metadata = city()

# Semantic

In [ ]:
import os
import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer, ColorMode

# Define input and output directories
input_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/train/images/"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/out/of_mask/coco"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/out/of_mask/city"
output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/out/of_mask/ade"


# predictor, metadata = coco()
# predictor, metadata = city()
predictor, metadata = ade()


# Ensure output base directory exists
os.makedirs(output_base_dir, exist_ok=True)
os.makedirs(input_dir, exist_ok=True)

# Get list of image files in input directory
image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for img_file in image_files:
    # 1) Load & resize
    img_path = os.path.join(input_dir, img_file)
    img = cv2.imread(img_path)                    # BGR
    img = imutils.resize(img, width=640)

    # 2) Extract base name and make output dir
    image_name = os.path.splitext(img_file)[0]
    output_dir = os.path.join(output_base_dir, image_name, "semantic")
    os.makedirs(output_dir, exist_ok=True)

    # 3) Run predictor
    task = "semantic"
    outputs = predictor(img, task)
    print(f"Predictor returned keys for {img_file}:", outputs.keys())

    # 4) Decode logits → per-pixel class map
    sem_seg_logits = outputs["sem_seg"]
    class_map = sem_seg_logits.detach().cpu().argmax(0).numpy()  # (H, W)

    # 5) Save binary masks and red overlay for each non-background class
    unique_ids = np.unique(class_map)
    class_info = {}
    for class_id in unique_ids:
        if class_id == 0:
            continue
        # Get human-readable name
        class_name = metadata.stuff_classes[class_id] \
                     if hasattr(metadata, "stuff_classes") else f"class_{class_id}"
        class_info[class_id] = class_name

        # Save binary mask
        mask = (class_map == class_id).astype(np.uint8) * 255
        mask_path = os.path.join(output_dir, f"{class_name}_id_{class_id}.png")
        cv2.imwrite(mask_path, mask)

        # Save red overlay visualization
        overlay = img.copy()
        colored_mask = np.zeros_like(img)
        colored_mask[:, :, 2] = mask  # Red channel
        overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)
        vis_path = os.path.join(output_dir, f"{class_name}_id_{class_id}_vis.jpg")
        cv2.imwrite(vis_path, overlay)

    print(f"Found {len(class_info)} unique segments in {img_file}: {class_info.values()}")

    # 6) Build one nice colored overlay via Detectron2's Visualizer
    viz = Visualizer(
        img[:, :, ::-1],                # convert BGR→RGB for the Visualizer
        metadata=metadata,
        instance_mode=ColorMode.SEGMENTATION
    )
    out_viz = viz.draw_sem_seg(class_map)

    # 7) Extract BGR overlay, convert to RGB for Matplotlib
    overlay_bgr = out_viz.get_image()           # BGR
    overlay_rgb = overlay_bgr[:, :, ::-1]       # → RGB

    # 8) Display with Matplotlib
    plt.figure(figsize=(19, 10))
    plt.imshow(overlay_rgb)
    plt.axis("off")
    plt.show()

    # 9) Save final overlay
    overlay_path = os.path.join(output_base_dir, image_name, f"{image_name}_semantic_overlay.jpg")
    cv2.imwrite(overlay_path, overlay_bgr)
    print(f"Saved colored overlay for {img_file} to {overlay_path}")

# Instance

In [ ]:
import os
import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer, ColorMode

# ─── 1. Define image directory and get all image files ────────────────────────
img_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/images/"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/coco"

output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/city"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/ade"

# predictor, metadata = coco()
predictor, metadata = city()
# predictor, metadata = ade()


img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
assert len(img_files) > 0, f"No images found in {img_dir}"

# ─── 2. Process each image ────────────────────────────────────────────────────
for img_file in img_files:
    img_path = os.path.join(img_dir, img_file)
    img = cv2.imread(img_path)
    assert img is not None, f"Failed to load {img_path}"
    img = imutils.resize(img, width=640)

    # ─── 3. Extract image name and create output directory ────────────────────
    image_name = os.path.splitext(img_file)[0]
    output_dir = os.path.join(output_base_dir, image_name, "instance")
    os.makedirs(output_dir, exist_ok=True)

    # ─── 4. Run instance-segmentation predictor ───────────────────────────────
    task = "instance"
    outputs = predictor(img, task)
    print(f"Predictor returned keys for {img_file}:", outputs.keys())

    # ─── 5. Unpack instance outputs ───────────────────────────────────────────
    instances    = outputs["instances"].to("cpu")
    pred_masks   = instances.pred_masks.numpy()    # (N, H, W)
    pred_classes = instances.pred_classes.numpy()  # (N,)
    scores       = instances.scores.numpy()        # (N,)

    # ─── 6. Save per-instance mask and overlay ────────────────────────────────
    class_info = {}
    for idx, (mask, cid, score) in enumerate(zip(pred_masks, pred_classes, scores)):
        class_name = metadata.thing_classes[cid]
        class_info[cid] = class_name

        # 6a) Save binary mask
        binary_mask = (mask.astype(np.uint8) * 255)
        mask_path = os.path.join(output_dir, f"{class_name}_id{cid}.png")
        cv2.imwrite(mask_path, binary_mask)

        # 6b) Save overlay visualization (red tint)
        overlay = img.copy()
        colored_mask = np.zeros_like(img)
        colored_mask[:, :, 2] = binary_mask
        overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)
        vis_path = os.path.join(output_dir, f"{class_name}_id_{cid}_vis.jpg")
        cv2.imwrite(vis_path, overlay)

    print(f"Found {len(class_info)} unique classes in {img_file}: {class_info.values()}")

    # ─── 7. Visualize full overlay with all instances ─────────────────────────
    viz = Visualizer(img[:, :, ::-1], metadata=metadata, instance_mode=ColorMode.SEGMENTATION)
    out_viz = viz.draw_instance_predictions(instances)
    overlay_bgr = out_viz.get_image()
    overlay_rgb = overlay_bgr[:, :, ::-1]

    # ─── 8. Show and save combined overlay ────────────────────────────────────
    plt.figure(figsize=(19, 10))
    plt.imshow(overlay_rgb)
    plt.axis("off")
    plt.show()

    combined_path = os.path.join(output_dir, f"{image_name}_instance_overlay.jpg")
    cv2.imwrite(combined_path, overlay_bgr)
    print(f"Saved colored overlay to {combined_path}")

# Panoptic

In [ ]:
import os
import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer, ColorMode

# 1) Define image directory and get all image files
# img_dir = "./seggpt/images/"
img_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/images/"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/coco"

output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/city"
# output_base_dir = "/home/sabbbir/Documents/of/Brats_FT/brats/tt/of_mask/ade"

# predictor, metadata = coco()
predictor, metadata = city()
# predictor, metadata = ade()
os.makedirs(output_base_dir, exist_ok=True)



img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for img_file in img_files:
    # 2) Load & resize
    img_path = os.path.join(img_dir, img_file)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Failed to load image: {img_path}")
        continue
    img = imutils.resize(img, width=640)

    # 3) Extract image name and make output dir
    image_name = os.path.splitext(os.path.basename(img_path))[0]

    output_dir = os.path.join(output_base_dir, image_name, "panoptic")
    os.makedirs(output_dir, exist_ok=True)

    # 4) Run predictor
    task = "panoptic"
    outputs = predictor(img, task)
    print(f"Processing {img_file} - Predictor returned keys:", outputs.keys())

    # 5) Get panoptic map
    panoptic_seg, segments_info = outputs["panoptic_seg"]
    panoptic_np = panoptic_seg.detach().cpu().numpy()
    class_info = {}

    # 6) Save masks and overlays
    for seg in segments_info:
        sid = seg["id"]
        cid = seg["category_id"]
        isthing = seg["isthing"]
        
        # Determine class name based on whether it's a thing or stuff class
        try:
            class_name = metadata.thing_classes[cid] if isthing else metadata.stuff_classes[cid]
        except:
            class_name = f"class_{cid}"
        
        class_info[sid] = class_name

        # Save binary mask
        mask = (panoptic_np == sid).astype(np.uint8) * 255
        mask_path = os.path.join(output_dir, f"{class_name}_id_{cid}.png")
        cv2.imwrite(mask_path, mask)
        # print(f"Saved binary mask to {mask_path}")

        # Save overlay visualization (red tint)
        overlay = img.copy()
        # Create a red mask (BGR format)
        colored_mask = np.zeros_like(img)
        colored_mask[:, :, 2] = mask  # Set red channel to the binary mask
        # Blend the original image with the red mask
        overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)
        vis_path = os.path.join(output_dir, f"{class_name}_id_{cid}_vis.jpg")
        success = cv2.imwrite(vis_path, overlay)
        # if success:
        #     print(f"Saved red overlay to {vis_path}")
        # else:
        #     print(f"Failed to save red overlay to {vis_path}")

    print(f"Found {len(class_info)} unique segments for {img_file}: {class_info.values()}")

    # 7) Visualize full overlay
    viz = Visualizer(img[:, :, ::-1], metadata=metadata, instance_mode=ColorMode.SEGMENTATION)
    out_viz = viz.draw_panoptic_seg_predictions(panoptic_seg.cpu(), segments_info)
    overlay_bgr = out_viz.get_image()
    overlay_rgb = overlay_bgr[:, :, ::-1]

    # 8) Show and save
    plt.figure(figsize=(19, 10))
    plt.imshow(overlay_rgb)
    plt.axis("off")
    plt.show()

    overlay_path = f"seggpt/masks/{image_name}/{image_name}_panoptic_overlay.jpg"
    cv2.imwrite(overlay_path, overlay_bgr)
    print(f"Saved colored overlay to {overlay_path}")

In [1]:
print("JJJ")

JJJ
